# ChromaDB RAG System: Complete Notebook
### From PDF Ingestion → Retrieval → RAG Answer Generation → Evaluation → Prompt Fine-Tuning

---
**What you'll learn in this notebook:**
| Step | Topic |
|------|-------|
| 1 | Install & Import dependencies |
| 2 | Load & chunk a PDF document |
| 3 | Generate embeddings with SentenceTransformer |
| 4 | Store vectors in ChromaDB |
| 5 | Visualise chunk distributions (Plotly) |
| 6 | Semantic similarity & retrieval visualisation |
| 7 | Build a RAG pipeline (retrieve → prompt → generate) |
| 8 | Evaluate answers with cosine similarity |
| 9 | Prompt fine-tuning & comparison |

> **Runtime tip:** Use a GPU runtime in Colab (`Runtime → Change runtime type → T4 GPU`) for faster model inference.


---
## 📦 Section 1 : Install Dependencies

We need:
- **chromadb** : vector database
- **sentence-transformers** : embedding model
- **pypdf + langchain** : PDF loading & splitting
- **transformers + torch** : LLM for generation
- **plotly** : interactive visualisations


In [ ]:
%pip install -q chromadb sentence-transformers pypdf langchain langchain-community langchain-text-splitters transformers accelerate torch plotly pandas numpy

---
## 🔧 Section 2 : Import Libraries

Import everything we'll need across the notebook.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sentence_transformers import SentenceTransformer, util
import chromadb

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


---
## 📄 Section 3 : Load a PDF Document

We use **PyPDFLoader** from LangChain to load a PDF.  
If you're running on **Google Colab**, use the file upload cell below.  
If you're running **locally**, set `PDF_PATH` to your file path.


In [ ]:
# ↪ Option A: Google Colab upload ↪
# Uncomment and run this block if you're on Colab
# from google.colab import files
# uploaded = files.upload()
# PDF_PATH = list(uploaded.keys())[0]

# ↪ Option B: Local path ↪
PDF_PATH = "/content/LLm_ML Interview questions - answers 43.pdf"   # ← change to your file

# Verify the file exists
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found at: {PDF_PATH}. Please upload or set the correct path.")

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"📄 PDF loaded: {PDF_PATH}")
print(f"📃 Total pages: {len(documents)}")
print(f"\n--- First 1000 characters of Page 1 ---\n")
print(documents[0].page_content[:1000])


📄 PDF loaded: /content/LLm_ML Interview questions - answers 43.pdf
📃 Total pages: 43

--- First 1000 characters of Page 1 ---

ML|LLMInterview 
Linearregression
isapopularsupervisedmachinelearningalgorithmusedforpredictingacontinuoustargetvariable(alsocalledthedependentvariable)basedononeormorepredictorvariables(independentvariables).Theprimarygoaloflinearregressionistofindthebest-fitlinearrelationshipbetweenthepredictorsandthetargetvariable.Here'showlinearregressionworks,includingtheobjectivefunctionandgradientdescent: ObjectiveFunction(CostFunction):Theobjectiveinlinearregressionistofindtheparameters(coefficients)ofthelinearmodelthatminimizethedifferencebetweenthepredictedvaluesandtheactualvaluesofthetargetvariable.Thisistypicallydoneusingacostfunction.ThemostcommoncostfunctionforlinearregressionistheMeanSquaredError(MSE)ortheSumofSquaredResiduals(SSR),whichisdefinedas:MSE=(1/2m)*Σ( yi-ŷi)^2Where:● MSE:MeanSquaredError,thecosttobeminimized.● m:Thenumberofdatapointsinthedataset.● yi:T

---
## ✂️ Section 4 : Text Chunking

Long documents can't be embedded as one blob : they need to be split into
manageable **chunks** so the embedding model can capture focused meaning.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `chunk_size` | 500 | Max characters per chunk |
| `chunk_overlap` | 100 | Characters shared between adjacent chunks (preserves context at boundaries) |


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"\n--- Sample Chunk (index 0) ---")
print(chunks[0].page_content)


✅ Total chunks created: 255

--- Sample Chunk (index 0) ---
ML|LLMInterview 
Linearregression


---
## 📊 Section 5 : Visualise Chunk Length Distribution

Understanding how chunk lengths are distributed helps us decide if `chunk_size`
is appropriate. Ideally, most chunks should be close to the target size.


In [ ]:
chunk_lengths = [len(c.page_content) for c in chunks]
chunk_pages   = [c.metadata.get("page", 0) for c in chunks]

df_chunks = pd.DataFrame({
    "chunk_index": range(len(chunks)),
    "length": chunk_lengths,
    "page": chunk_pages
})

# ↪ Histogram of chunk lengths ↪
fig1 = px.histogram(
    df_chunks, x="length", nbins=30,
    title="📏 Distribution of Document Chunk Lengths",
    labels={"length": "Characters per Chunk", "count": "Frequency"},
    color_discrete_sequence=["#7C3AED"],
    template="plotly_dark"
)
fig1.add_vline(x=np.mean(chunk_lengths), line_dash="dash", line_color="#F59E0B",
               annotation_text=f"Mean: {np.mean(chunk_lengths):.0f}", annotation_position="top right")
fig1.update_layout(bargap=0.05)
fig1.show()

# ↪ Chunks per page scatter ↪
fig2 = px.scatter(
    df_chunks, x="chunk_index", y="length", color="page",
    title="📄 Chunk Length by Document Position",
    labels={"chunk_index": "Chunk Index", "length": "Length (chars)", "page": "Page"},
    color_continuous_scale="Viridis",
    template="plotly_dark"
)
fig2.show()

print(f"📊 Stats : Min: {min(chunk_lengths)} | Max: {max(chunk_lengths)} | Mean: {np.mean(chunk_lengths):.0f} | Median: {np.median(chunk_lengths):.0f}")


📊 Stats — Min: 15 | Max: 500 | Mean: 348 | Median: 360


---
## 🔢 Section 6 : Generate Embeddings

An **embedding** is a dense numeric vector that captures the *meaning* of text.
Semantically similar texts → similar vectors → close in vector space.

We use **`BAAI/bge-small-en-v1.5`** : a small, fast, high-quality English embedding model.


In [ ]:
print("⏳ Loading embedding model (BAAI/bge-small-en-v1.5)...")
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print(f"✅ Model loaded! Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


⏳ Loading embedding model (BAAI/bge-small-en-v1.5)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded! Embedding dimension: 384


### 6a : Cosine Similarity: Semantic Matching

Cosine similarity ranges from **0** (unrelated) to **1** (identical meaning).
Let's verify the model works correctly on known similar/dissimilar pairs.


In [ ]:
test_sentences = [
    "What is overfitting?",                       # 0 : query
    "Explain overfitting in machine learning",    # 1 : very similar
    "How to prevent a model from overfitting",    # 2 : related
    "What is underfitting?",                      # 3 : somewhat related
    "How to cook pizza at home?",                 # 4 : unrelated
    "History of ancient Rome",                    # 5 : unrelated
]

embeddings = embedding_model.encode(test_sentences, convert_to_tensor=True)

query_emb = embeddings[0]
scores = [util.cos_sim(query_emb, embeddings[i]).item() for i in range(len(test_sentences))]

df_sim = pd.DataFrame({
    "Sentence": test_sentences,
    "Cosine Similarity": scores,
    "Category": ["Query", "Highly Similar", "Related", "Somewhat Related", "Unrelated", "Unrelated"]
})

fig3 = px.bar(
    df_sim, x="Sentence", y="Cosine Similarity",
    color="Category",
    title="🔍 Cosine Similarity vs Query: 'What is overfitting?'",
    color_discrete_map={
        "Query": "#6366F1",
        "Highly Similar": "#10B981",
        "Related": "#84CC16",
        "Somewhat Related": "#F59E0B",
        "Unrelated": "#EF4444"
    },
    template="plotly_dark",
    text="Cosine Similarity"
)
fig3.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig3.update_layout(xaxis_tickangle=-25, showlegend=True, height=500)
fig3.show()

df_sim


,Sentence,Cosine Similarity,Category
0,What is overfitting?,1.000000,Query
1,Explain overfitting in machine learning,0.846092,Highly Similar
2,How to prevent a model from overfitting,0.752662,Related
3,What is underfitting?,0.848871,Somewhat Related
4,How to cook pizza at home?,0.358590,Unrelated
5,History of ancient Rome,0.357626,Unrelated


In [ ]:
scores

[1.0,
 0.8460916876792908,
 0.7526618242263794,
 0.8488713502883911,
 0.3585895895957947,
 0.35762593150138855]

---
## 🗄️ Section 7 : Store Embeddings in ChromaDB

**ChromaDB** is an open-source vector database. We:
1. Create an in-memory client
2. Create a named **collection** (like a table)
3. Insert all chunks with their embeddings and metadata


In [ ]:
# In-memory ChromaDB (no disk persistence needed for this demo)
client = chromadb.Client()

# Delete collection if it already exists (useful for re-runs)
try:
    client.delete_collection("rag_collection")
except:
    pass

collection = client.create_collection(
    name="rag_collection",
    metadata={"hnsw:space": "cosine"}   # Use cosine distance
)

print("⏳ Embedding and storing chunks...")
for i, chunk in enumerate(chunks):
    embedding = embedding_model.encode(chunk.page_content).tolist()
    collection.add(
        ids=[str(i)],
        documents=[chunk.page_content],
        embeddings=[embedding],
        metadatas=[chunk.metadata]
    )
    if (i + 1) % 50 == 0:
        print(f"  → {i+1}/{len(chunks)} chunks stored...")

print(f"\n✅ Done! Total documents in collection: {collection.count()}")


⏳ Embedding and storing chunks...
  → 50/255 chunks stored...
  → 100/255 chunks stored...
  → 150/255 chunks stored...
  → 200/255 chunks stored...
  → 250/255 chunks stored...

✅ Done! Total documents in collection: 255


---
## 🔎 Section 8 : Retrieval & Similarity Visualisation

Now we query ChromaDB with a question. It returns the **top-k most relevant chunks**
ranked by cosine similarity (lower distance = more similar).


In [ ]:
def retrieve(question: str, n_results: int = 5) -> dict:
    """Retrieve top-n relevant chunks from ChromaDB for a given question."""
    q_emb = embedding_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=n_results,
        include=["documents", "distances", "metadatas"]
    )
    return results

# ↪ Query ↪
query = "What is overfitting?"
results = retrieve(query, n_results=5)

print(f"Query: '{query}'\n{'='*60}")
for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
    similarity = 1 - dist   # cosine distance → similarity
    print(f"\n🔹 Rank {i+1} | Similarity: {similarity:.4f}")
    print(doc[:300])
    print("-"*60)


Query: 'What is overfitting?'

🔹 Rank 1 | Similarity: 0.7157
Whatisoverfitting,andhowdoesboostingaddressit?● Overfittingoccurswhenamodellearnsthetrainingdatatoowell,includingnoise.Boostingcanmitigateoverfittingbecauseitprioritizesdifficult-to-classifyexamplesandgraduallycorrectsmisclassifications,leadingtoamoregeneralizedmodel. Canboostingmodelshandlenoisydat
------------------------------------------------------------

🔹 Rank 2 | Similarity: 0.6916
Howdoyoupreventoverfittinginastackedensemble?Overfittingcanbecontrolledbyusingcross-validationwhencreatingthemeta-features.Insteadofusingtheentiredatasetfortrainingthemeta-model,youcanuseaportionofitandvalidateontherest.Additionally,youcanuseregularizationtechniquesonthemeta-model.4.
------------------------------------------------------------

🔹 Rank 3 | Similarity: 0.6353
● Highvariancecanleadtooverfitting,wherethemodelistoocomplexandfitsthetrainingdatatooclosely,failingtogeneralizetonew,unseendata.● Complexmodels,suchasdeepneuralnetworks

In [ ]:
# ↪ Retrieval similarity bar chart ↪
distances = results["distances"][0]
similarities = [1 - d for d in distances]
snippets = [doc[:60] + "..." for doc in results["documents"][0]]

df_ret = pd.DataFrame({
    "Rank": [f"Rank {i+1}" for i in range(len(similarities))],
    "Similarity": similarities,
    "Snippet": snippets
})

fig4 = px.bar(
    df_ret, x="Rank", y="Similarity",
    text="Similarity",
    hover_data=["Snippet"],
    title=f"📈 Retrieval Similarity Scores : Query: '{query}'",
    color="Similarity",
    color_continuous_scale="Tealgrn",
    template="plotly_dark"
)
fig4.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig4.update_layout(coloraxis_showscale=False, height=450)
fig4.show()


In [ ]:
# ↪ Multi-query comparison ↪
multi_queries = [
    "What is overfitting?",
    "How does gradient descent work?",
    "Explain cross-validation",
    "What is regularization?",
    "Describe a confusion matrix",
]

rows = []
for q in multi_queries:
    res = retrieve(q, n_results=3)
    for rank, (dist) in enumerate(res["distances"][0], 1):
        rows.append({"Query": q[:40], "Rank": f"Rank {rank}", "Similarity": round(1 - dist, 4)})

df_multi = pd.DataFrame(rows)

fig5 = px.bar(
    df_multi, x="Query", y="Similarity", color="Rank",
    barmode="group",
    title="🔍 Retrieval Quality Across Multiple Queries",
    template="plotly_dark",
    color_discrete_sequence=["#7C3AED", "#06B6D4", "#10B981"]
)
fig5.update_layout(xaxis_tickangle=-20, height=500)
fig5.show()


---
## 🤖 Section 9 : RAG Pipeline: Retrieve → Prompt → Generate

**Retrieval-Augmented Generation (RAG)** has three steps:
1. **Retrieve** : find relevant context from ChromaDB
2. **Augment** : inject context into a prompt
3. **Generate** : pass the prompt to an LLM for an answer

We use **`distilgpt2`** here (fast, CPU-friendly). Swap with `microsoft/Phi-3-mini-4k-instruct` on a GPU for much better quality.


In [ ]:
from transformers import pipeline

print("⏳ Loading generation model (distilgpt2)...")
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    pad_token_id=50256
)
print("✅ Generator ready!")


⏳ Loading generation model (distilgpt2)...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Generator ready!


In [ ]:
def build_prompt(question: str, context_chunks: list, style: str = "standard") -> str:
    """
    Build a RAG prompt. 'style' controls the instruction format:
      - 'standard'    : basic instruction
      - 'concise'     : ask for a short answer
      - 'step_by_step': ask for numbered reasoning
      - 'eli5'        : explain like I'm 5
    """
    context = "\n\n".join(context_chunks)

    instructions = {
        "standard": "Use the context below to answer the question accurately.",
        "concise":  "Use the context below. Answer in 2-3 sentences only. Be precise.",
        "step_by_step": "Use the context below. Answer with numbered steps and clear reasoning.",
        "eli5": "Use the context below. Explain the answer simply, as if to a complete beginner."
    }

    instruction = instructions.get(style, instructions["standard"])

    prompt = f"""{instruction}

Context:
{context}

Question: {question}

Answer:"""
    return prompt


def ask_rag(question: str, n_results: int = 3, style: str = "standard", max_new_tokens: int = 120) -> dict:
    """Full RAG pipeline: retrieve → build prompt → generate."""
    retrieved = retrieve(question, n_results=n_results)
    context_chunks = retrieved["documents"][0]
    distances = retrieved["distances"][0]

    prompt = build_prompt(question, context_chunks, style=style)

    response = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    full_text = response[0]["generated_text"]

    # Extract only the answer part (after "Answer:")
    answer = full_text.split("Answer:")[-1].strip()

    return {
        "question": question,
        "answer": answer,
        "context_chunks": context_chunks,
        "similarities": [round(1 - d, 4) for d in distances],
        "prompt": prompt,
        "style": style
    }

# ↪ Test it ↪
result = ask_rag("What is overfitting?", style="standard")
print(f"Question : {result['question']}")
print(f"Style    : {result['style']}")
print(f"\nAnswer:\n{result['answer']}")


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question : What is overfitting?
Style    : standard

Answer:
The answer is that the model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is


---
## 📐 Section 10 : Evaluation

We evaluate generated answers using **Semantic Similarity** against expected answers.
This is the approach used by frameworks like **Ragas** and **DeepEval**.

**Metric:** Cosine similarity between the embedding of the *generated answer*
and the embedding of the *expected (reference) answer*.  
Scale: 0 (no overlap) → 1 (perfect match).


In [ ]:
def evaluate_answer(generated: str, expected: str) -> float:
    """Return cosine similarity between generated and expected answer embeddings."""
    gen_emb = embedding_model.encode(generated, convert_to_tensor=True)
    exp_emb = embedding_model.encode(expected, convert_to_tensor=True)
    return round(util.cos_sim(gen_emb, exp_emb).item(), 4)


# ↪ Evaluation dataset ↪
eval_set = [
    {
        "question": "What is overfitting?",
        "expected": "Overfitting occurs when a model learns the noise in training data and performs poorly on unseen data because it memorises rather than generalises."
    },
    {
        "question": "What is cross-validation?",
        "expected": "Cross-validation is a technique to assess model performance by splitting data into multiple train/test folds and averaging the results."
    },
    {
        "question": "What is regularization?",
        "expected": "Regularization is a technique to reduce overfitting by adding a penalty term to the loss function, discouraging overly complex models."
    },
    {
        "question": "What is a confusion matrix?",
        "expected": "A confusion matrix is a table showing predicted vs actual class labels, used to evaluate classification model performance."
    },
    {
        "question": "How does gradient descent work?",
        "expected": "Gradient descent is an optimisation algorithm that iteratively adjusts model parameters in the direction of the negative gradient of the loss function to minimise error."
    },
]

print("⏳ Running RAG evaluation pipeline...")
eval_results = []

for item in eval_set:
    rag = ask_rag(item["question"], style="standard", max_new_tokens=100)
    score = evaluate_answer(rag["answer"], item["expected"])
    eval_results.append({
        "Question": item["question"],
        "Generated Answer": rag["answer"][:120] + "...",
        "Similarity Score": score,
        "Top Retrieval Sim": rag["similarities"][0] if rag["similarities"] else 0
    })
    print(f"  ✓ {item['question'][:50]} → Score: {score:.4f}")

df_eval = pd.DataFrame(eval_results)
print("\n✅ Evaluation complete!")
df_eval


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏳ Running RAG evaluation pipeline...


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✓ What is overfitting? → Score: 0.5325


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✓ What is regularization? → Score: 0.7543


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✓ What is a confusion matrix? → Score: 0.7375
  ✓ How does gradient descent work? → Score: 0.8211

✅ Evaluation complete!


,Question,Generated Answer,Similarity Score,Top Retrieval Sim
0,What is overfitting?,The answer is that the model is not a model.\n...,0.5325,0.7157
1,What is cross-validation?,Theoptimalalphavalueistypicallyfoundthroughtec...,0.7011,0.7809
2,What is regularization?,Q2.Howdoyouincreasetheaccuracyofaregressionmod...,0.7543,0.7254
3,What is a confusion matrix?,The confusion matrix is a matrix that is a mat...,0.7375,0.6645
4,How does gradient descent work?,The gradient descent algorithm is based on the...,0.8211,0.6990


In [ ]:
# ↪ Evaluation radar / bar chart ↪
fig6 = px.bar(
    df_eval, x="Question", y="Similarity Score",
    text="Similarity Score",
    color="Similarity Score",
    color_continuous_scale="RdYlGn",
    range_color=[0, 1],
    title="📊 RAG Answer Quality : Semantic Similarity vs Expected Answers",
    template="plotly_dark",
    labels={"Question": "Evaluation Question"}
)
fig6.add_hline(y=0.7, line_dash="dash", line_color="#F59E0B",
               annotation_text="Good threshold (0.7)", annotation_position="top left")
fig6.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig6.update_layout(xaxis_tickangle=-20, height=500, coloraxis_showscale=False)
fig6.show()

# ↪ Retrieval vs Answer quality scatter ↪
fig7 = px.scatter(
    df_eval, x="Top Retrieval Sim", y="Similarity Score",
    text="Question",
    title="🔗 Retrieval Quality vs Answer Quality",
    labels={"Top Retrieval Sim": "Top Retrieved Chunk Similarity", "Similarity Score": "Answer Similarity"},
    template="plotly_dark",
    size=[20]*len(df_eval),
    color="Similarity Score",
    color_continuous_scale="Tealgrn"
)
fig7.update_traces(textposition="top center")
fig7.update_layout(height=500)
fig7.show()


---
## 🎛️ Section 11 : Prompt Fine-Tuning & Style Comparison

The **same RAG system** can produce very different answers depending on how the
prompt is written. Here we compare 4 prompt styles on the same question and
measure which produces the best semantic alignment with the expected answer.

| Style | Instruction Strategy |
|-------|---------------------|
| `standard` | Basic: "Use context to answer" |
| `concise` | "Answer in 2-3 sentences only" |
| `step_by_step` | "Answer with numbered steps" |
| `eli5` | "Explain as if to a beginner" |


In [ ]:
# ↪ Compare prompt styles on a single question ↪
test_question = "What is overfitting?"
expected = "Overfitting occurs when a model learns the noise in training data and performs poorly on unseen data because it memorises rather than generalises."

styles = ["standard", "concise", "step_by_step", "eli5"]
style_results = []

print(f"Question: '{test_question}'\n{'='*60}")

for style in styles:
    res = ask_rag(test_question, style=style, max_new_tokens=100)
    score = evaluate_answer(res["answer"], expected)
    style_results.append({
        "Style": style,
        "Answer": res["answer"],
        "Similarity Score": score
    })
    print(f"\n🎨 Style: [{style.upper()}]")
    print(f"   Answer: {res['answer'][:200]}")
    print(f"   Similarity to expected: {score:.4f}")

df_styles = pd.DataFrame(style_results)


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: 'What is overfitting?'


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎨 Style: [STANDARD]
   Answer: The answer is that the model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
   Similarity to expected: 0.5325


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎨 Style: [CONCISE]
   Answer: Whatisoverfitting isoverfitting,andhowdoesboostingaddressit?● Overfittingcanmitigateoverfittingbecauseitprioritizesdifficult-to-classifyexamplesandgraduallycorrectsmisclassifications,leadingtoamoregen
   Similarity to expected: 0.6890


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🎨 Style: [STEP_BY_STEP]
   Answer: Whatisoverfitting isoverfitting,andhowdoesboostingaddressit?● Overfittingoccurswhenamodellearnsthetrainingdatatoowell,includingnoise.Boostingcanmitigateoverfittingbecauseitprioritizesdifficult-to-clas
   Similarity to expected: 0.7190

🎨 Style: [ELI5]
   Answer: The answer is that the model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
The model is not a model.
   Similarity to expected: 0.5325


In [ ]:
# ↪ Prompt style comparison chart ↪
fig8 = px.bar(
    df_styles, x="Style", y="Similarity Score",
    text="Similarity Score",
    color="Style",
    title=f"🎛️ Prompt Style Comparison : '{test_question}'",
    template="plotly_dark",
    color_discrete_sequence=["#7C3AED", "#06B6D4", "#10B981", "#F59E0B"]
)
fig8.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig8.update_layout(showlegend=False, height=450)
fig8.show()

# Best style
best = df_styles.loc[df_styles["Similarity Score"].idxmax()]
print(f"\n🏆 Best prompt style: [{best['Style'].upper()}] with similarity score: {best['Similarity Score']:.4f}")



🏆 Best prompt style: [STEP_BY_STEP] with similarity score: 0.7190


In [ ]:
# ↪ Full prompt inspection : view what the model actually receives ↪
best_style = best["Style"]
res_best = ask_rag(test_question, style=best_style, max_new_tokens=100)

print(f"=== FULL PROMPT (style: {best_style}) ===\n")
print(res_best["prompt"])
print(f"\n=== GENERATED ANSWER ===\n{res_best['answer']}")


[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== FULL PROMPT (style: step_by_step) ===

Use the context below. Answer with numbered steps and clear reasoning.

Context:
Whatisoverfitting,andhowdoesboostingaddressit?● Overfittingoccurswhenamodellearnsthetrainingdatatoowell,includingnoise.Boostingcanmitigateoverfittingbecauseitprioritizesdifficult-to-classifyexamplesandgraduallycorrectsmisclassifications,leadingtoamoregeneralizedmodel. Canboostingmodelshandlenoisydata?● Boostingmodelscanhandlenoisydatatosomeextent.Theyarerobusttonoisydata,butextremenoisecanstillimpacttheirperformance.Datapreprocessingandcleaningareoftenrequiredforbetterresults.

Howdoyoupreventoverfittinginastackedensemble?Overfittingcanbecontrolledbyusingcross-validationwhencreatingthemeta-features.Insteadofusingtheentiredatasetfortrainingthemeta-model,youcanuseaportionofitandvalidateontherest.Additionally,youcanuseregularizationtechniquesonthemeta-model.4.

● Highvariancecanleadtooverfitting,wherethemodelistoocomplexandfitsthetrainingdatatooclosely,failingtogener

In [ ]:
# ↪ Multi-question × Multi-style heatmap ↪
heatmap_questions = [
    "What is overfitting?",
    "What is cross-validation?",
    "What is regularization?",
]
heatmap_expected = [
    "Overfitting occurs when a model learns noise from training data and performs poorly on unseen data.",
    "Cross-validation splits data into folds to evaluate model performance more robustly.",
    "Regularization adds a penalty to the loss function to reduce model complexity and overfitting.",
]

heatmap_data = {}
for style in styles:
    col = []
    for q, exp in zip(heatmap_questions, heatmap_expected):
        res = ask_rag(q, style=style, max_new_tokens=80)
        score = evaluate_answer(res["answer"], exp)
        col.append(score)
    heatmap_data[style] = col

df_heat = pd.DataFrame(heatmap_data, index=[q[:35] for q in heatmap_questions])

fig9 = px.imshow(
    df_heat,
    text_auto=".3f",
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=1,
    title="🌡️ Prompt Style × Question : Semantic Similarity Heatmap",
    labels={"x": "Prompt Style", "y": "Question", "color": "Similarity"},
    template="plotly_dark",
    aspect="auto"
)
fig9.update_layout(height=400)
fig9.show()

print("\n📌 Best prompt style per question:")
print(df_heat.idxmax(axis=1).to_string())


[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs


📌 Best prompt style per question:
What is overfitting?         step_by_step
What is cross-validation?        standard
What is regularization?          standard


---
## ✅ Section 12 : Summary & Next Steps

### What we built:
```
PDF  →  Chunks  →  Embeddings  →  ChromaDB
                                       ↓
              Question  →  Query  →  Top-k Chunks
                                       ↓
                            Prompt  →  LLM  →  Answer
                                       ↓
                              Evaluate (Cosine Sim)
```

### Key takeaways:
| Concept | Insight |
|---------|---------|
| **Chunk size** | 500 chars with 100 overlap works well for Q&A tasks |
| **Embedding model** | `BAAI/bge-small-en-v1.5` is fast and high-quality |
| **Retrieval** | Cosine similarity ≥ 0.75 usually gives good context |
| **Prompt style** | Small changes in instruction wording measurably change answer quality |
| **Evaluation** | Semantic similarity is a fast, reference-based metric |

### Next steps to level up:
1. **Swap the LLM** : Replace `distilgpt2` with `Phi-3-mini`, `Llama-3`, or call the **OpenAI / Anthropic API**
2. **Persistent ChromaDB** : Use `chromadb.PersistentClient(path="./chroma_db")` to save across sessions
3. **Metadata filtering** : Filter by page, section, or source when querying
4. **Advanced evaluation** : Integrate **Ragas** (`pip install ragas`) for faithfulness, context precision, answer relevance
5. **Hybrid search** : Combine dense embeddings with BM25 sparse retrieval for better recall
